# LDM — YOLO11s one-class litter model (AUG100 → `LDM_TRAINING_READY`)

Trains a **separate one-class `litter` model** on Google Colab GPU. It does **not** touch or replace the production
`v3_store_cctv` model (10 classes); the class mappings are incompatible and none is invented.

**Before running**
1. On the LDM PC: `python -m training.prepare_training_ready` → `C:\SHREE\DATASET\LDM_TRAINING_READY.zip`.
2. Upload `LDM_TRAINING_READY.zip` to Google Drive → `MyDrive/LDM/LDM_TRAINING_READY.zip`.
3. *(optional, for the comparison table)* upload `models/litter_yolo/v3_store_cctv/best.pt` to `MyDrive/LDM/v3_store_cctv_best.pt`.
4. Colab: **Runtime → Change runtime type → GPU** (T4 or better), then **Runtime → Run all**.

**Outputs** (all on Drive under `MyDrive/LDM/runs/<RUN_NAME>/`): checkpoints every `SAVE_PERIOD` epochs, `best.pt`, `last.pt`,
curves, confusion matrices, the CSV reports, and `ldm_model_<MODEL_VERSION>.zip` to install in LDM with
`python -m training.install_trained_model --package <zip>` (never activates it).

**Honesty notes:** all 100 images are synthetic transforms of two reference frames; test = the *other* scene. Metrics measure
cross-scene transfer inside one synthetic generator, not real CCTV accuracy. Labels were AI-reviewed, not human-annotated.

In [ ]:
# ===================================================== 1. Configuration (edit here only)
import os
SMOKE = os.environ.get('LDM_SMOKE') == '1'          # local CPU smoke test of this notebook (not a real training)

DRIVE_ROOT    = '/content/drive/MyDrive/LDM'          # Drive folder holding the dataset zip
DATASET_ZIP   = 'LDM_TRAINING_READY.zip'
V3_WEIGHTS    = 'v3_store_cctv_best.pt'               # optional, in DRIVE_ROOT: class-agnostic comparison only
MODEL_VERSION = 'v4_aug100_litter1c'                  # new, separate version folder name in LDM
RUN_NAME      = MODEL_VERSION + '_run1'               # must be new: an existing run is never overwritten (set RESUME)
RESUME        = False                                 # True: continue RUN_NAME from its last.pt after a disconnect

START_WEIGHTS = 'yolo11s.pt'                          # COCO-pretrained YOLO11s (same base as v1->v3)
START_SHA256  = '85a76fe86dd8afe384648546b56a7a78580c7cb7b404fc595f97969322d502d5'  # models/pretrained/yolo11s.pt in LDM
EPOCHS        = 100
PATIENCE      = 25
BATCH         = 'auto'                                # 'auto' = chosen from GPU memory below, or an int
IMG_SIZE      = 640                                   # LDM inference default. 60% of boxes are <16 px at 1280x720:
                                                      # 960 or 1280 is worth a second run (compare in model_comparison.csv)
OPTIMIZER     = 'SGD'                                 # explicit: 'auto' silently ignores LR0
LR0           = 0.01
LRF           = 0.01
MOMENTUM      = 0.937
WEIGHT_DECAY  = 0.0005
WARMUP_EPOCHS = 3
SEED          = 0
SAVE_PERIOD   = 10
EVAL_CONF     = 0.25                                  # confidence for the FP/FN report (LDM weak threshold is 0.30)
MATCH_IOU     = 0.5
# realistic CCTV augmentation only: fixed camera, no vertical flip, little rotation, no mixup/copy-paste
AUG = dict(hsv_h=0.01, hsv_s=0.4, hsv_v=0.35, degrees=2.0, translate=0.1, scale=0.3, shear=0.0, perspective=0.0002,
           flipud=0.0, fliplr=0.5, mosaic=0.7, close_mosaic=10, mixup=0.0, copy_paste=0.0, erasing=0.0)

if SMOKE:   # tiny settings: proves the code path only
    EPOCHS, PATIENCE, BATCH, IMG_SIZE, SAVE_PERIOD = 1, 1, 2, 320, -1

In [ ]:
# ===================================================== 2. Google Drive / local paths
import sys, pathlib
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    WORK = pathlib.Path('/content/work')
else:   # local smoke test: LDM_LOCAL_ROOT plays the role of DRIVE_ROOT
    DRIVE_ROOT = os.environ['LDM_LOCAL_ROOT']
    WORK = pathlib.Path(os.environ.get('LDM_LOCAL_WORK', pathlib.Path(DRIVE_ROOT) / 'work'))
DRIVE = pathlib.Path(DRIVE_ROOT)
RUNS = DRIVE / 'runs'
RUNS.mkdir(parents=True, exist_ok=True)
WORK.mkdir(parents=True, exist_ok=True)
print('Drive folder:', DRIVE, '| runs ->', RUNS)

In [ ]:
# ===================================================== 3. Software (pinned to the LDM PC's Ultralytics version)
if IN_COLAB:
    !pip -q install ultralytics==8.4.173
import torch, ultralytics, platform
print('Python', platform.python_version(), '| torch', torch.__version__, '| CUDA build', torch.version.cuda,
      '| ultralytics', ultralytics.__version__)

In [ ]:
# ===================================================== 4. GPU check
if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    GPU_GB = props.total_memory / 2**30
    DEVICE = 0
    print(f'GPU: {props.name}  {GPU_GB:.1f} GB')
else:
    GPU_GB, DEVICE = 0.0, 'cpu'
    if IN_COLAB:
        raise SystemExit('No GPU: Runtime -> Change runtime type -> GPU, then run again.')
    print('CPU only (local smoke test)')
if BATCH == 'auto':   # YOLO11s, AMP: ~0.45 GB per image at 640 (x2.25 at 960, x4 at 1280), 20% headroom
    per_img = 0.45 * (IMG_SIZE / 640) ** 2
    BATCH = int(max(2, min(32, (GPU_GB * 0.8) // per_img)))
print('batch', BATCH, '| imgsz', IMG_SIZE)

In [ ]:
# ===================================================== 5. Dataset: unpack, validate YAML, class mapping and labels
import zipfile, yaml, shutil, json, csv
DS = WORK / 'LDM_TRAINING_READY'
if not DS.exists():
    src = DRIVE / DATASET_ZIP
    if src.is_file():
        with zipfile.ZipFile(src) as z:
            z.extractall(WORK)
    elif (DRIVE / 'LDM_TRAINING_READY').is_dir():
        shutil.copytree(DRIVE / 'LDM_TRAINING_READY', DS)
    else:
        raise SystemExit(f'Upload {DATASET_ZIP} to {DRIVE} first')
cfg = yaml.safe_load((DS / 'data.yaml').read_text())
assert cfg.get('names') in ({0: 'litter'}, ['litter']), f"unexpected class mapping {cfg.get('names')}"
assert int(cfg.get('nc', 1)) == 1
cfg['path'] = str(DS)
DATA_YAML = WORK / 'data_abs.yaml'
DATA_YAML.write_text(yaml.safe_dump(cfg, sort_keys=False))

IMG_EXT = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}
def split_files(split):
    return sorted(p for p in (DS / 'images' / split).iterdir() if p.suffix.lower() in IMG_EXT)
def read_labels(img_path):
    lp = DS / 'labels' / img_path.parent.name / (img_path.stem + '.txt')
    rows = [l.split() for l in lp.read_text().splitlines() if l.strip()] if lp.is_file() else None
    return rows
problems, stats = [], {}
for split in ('train', 'val', 'test'):
    files = split_files(split)
    nbox = 0
    for f in files:
        rows = read_labels(f)
        if rows is None:
            problems.append(f'{split}/{f.name}: no label file'); continue
        for r in rows:
            c, *xywh = r
            v = [float(x) for x in xywh]
            if c != '0' or len(v) != 4 or min(v) < 0 or max(v) > 1 or v[2] <= 0 or v[3] <= 0:
                problems.append(f'{split}/{f.name}: bad row {r}')
        nbox += len(rows)
    stats[split] = (len(files), nbox)
print('images/boxes per split:', stats)
assert not problems, problems[:10]
IGNORE = json.loads((DS / 'ignore_regions.json').read_text())['regions'] if (DS / 'ignore_regions.json').is_file() else {}
print('dataset OK; ignore regions for', len(IGNORE), 'images')

In [ ]:
# ===================================================== 6. Sample images with ground truth
import cv2, matplotlib
if not IN_COLAB:
    matplotlib.use('Agg')
import matplotlib.pyplot as plt
def gt_boxes(img_path, w, h):
    return [[(float(cx) - float(bw) / 2) * w, (float(cy) - float(bh) / 2) * h, (float(cx) + float(bw) / 2) * w,
             (float(cy) + float(bh) / 2) * h] for _, cx, cy, bw, bh in (read_labels(img_path) or [])]
samples = split_files('train')[:2] + split_files('test')[:2]
fig, axes = plt.subplots(2, 2, figsize=(16, 9))
for ax, p in zip(axes.flat, samples):
    im = cv2.imread(str(p)); h, w = im.shape[:2]
    for x1, y1, x2, y2 in gt_boxes(p, w, h):
        cv2.rectangle(im, (int(x1), int(y1)), (int(x2), int(y2)), (0, 255, 0), 2)
    for x1, y1, x2, y2 in IGNORE.get(p.name, []):
        cv2.rectangle(im, (int(x1), int(y1)), (int(x2), int(y2)), (128, 128, 128), 1)
    ax.imshow(cv2.cvtColor(im, cv2.COLOR_BGR2RGB)); ax.set_title(f'{p.parent.name}/{p.name}'); ax.axis('off')
plt.tight_layout(); plt.savefig(WORK / 'samples.png', dpi=80); plt.show() if IN_COLAB else None

In [ ]:
# ===================================================== 7. Starting weights (verified)
import hashlib
from ultralytics import YOLO
def sha256(p):
    h = hashlib.sha256()
    with open(p, 'rb') as fh:
        for b in iter(lambda: fh.read(1 << 20), b''): h.update(b)
    return h.hexdigest()
sw = os.environ.get('LDM_START_WEIGHTS') or START_WEIGHTS
if not pathlib.Path(sw).is_file():
    YOLO(sw)                                    # Ultralytics downloads the official file into the working dir
START_PATH = pathlib.Path(sw).resolve()
got = sha256(START_PATH)
print('start weights', START_PATH.name, got)
if START_SHA256 and got != START_SHA256:
    print('WARNING: start weights differ from the file LDM used before (', START_SHA256[:12], '). Recorded in metadata.')

In [ ]:
# ===================================================== 8. Train (checkpoints go straight to Drive)
import time, datetime as dt
RUN_DIR = RUNS / RUN_NAME
t0 = time.time()
if RESUME:
    assert (RUN_DIR / 'weights' / 'last.pt').is_file(), 'nothing to resume'
    YOLO(str(RUN_DIR / 'weights' / 'last.pt')).train(resume=True)
else:
    assert not RUN_DIR.exists(), f'{RUN_DIR} exists: choose a new RUN_NAME or set RESUME = True'
    model = YOLO(str(START_PATH))
    model.train(data=str(DATA_YAML), epochs=EPOCHS, patience=PATIENCE, batch=BATCH, imgsz=IMG_SIZE, device=DEVICE,
                optimizer=OPTIMIZER, lr0=LR0, lrf=LRF, momentum=MOMENTUM, weight_decay=WEIGHT_DECAY,
                warmup_epochs=WARMUP_EPOCHS, seed=SEED, deterministic=True, project=str(RUNS),
                name=RUN_NAME, exist_ok=False, save_period=SAVE_PERIOD, plots=True, cache=False,
                workers=2 if IN_COLAB else 0, amp=DEVICE != 'cpu', **AUG)
TRAIN_MIN = round((time.time() - t0) / 60, 1)
BEST, LAST = RUN_DIR / 'weights' / 'best.pt', RUN_DIR / 'weights' / 'last.pt'
assert BEST.is_file(), 'training produced no best.pt'
print('trained in', TRAIN_MIN, 'min ->', BEST)

In [ ]:
# ===================================================== 9. Validate (val) and evaluate on the independent test scene
best = YOLO(str(BEST))
EVAL = {}
for split in ('val', 'test'):
    m = best.val(data=str(DATA_YAML), split=split, imgsz=IMG_SIZE, batch=BATCH, device=DEVICE, plots=True,
                 project=str(RUN_DIR), name=f'eval_{split}', exist_ok=True, conf=0.001, iou=0.6)
    p, r = float(m.box.mp), float(m.box.mr)
    EVAL[split] = dict(precision=round(p, 4), recall=round(r, 4), f1=round(2 * p * r / (p + r), 4) if p + r else 0.0,
                       mAP50=round(float(m.box.map50), 4), mAP50_95=round(float(m.box.map), 4),
                       images=stats[split][0], boxes=stats[split][1])
    print(split, EVAL[split])
print('confusion matrices / PR curves:', [str(RUN_DIR / f'eval_{s}') for s in ('val', 'test')])

In [ ]:
# ===================================================== 10. Reports: evaluation_summary, per_class_metrics, training_summary
REPORTS = RUN_DIR / 'reports'
REPORTS.mkdir(exist_ok=True)
def write_csv(name, rows, fields=None):
    fields = fields or list(rows[0].keys())
    with open(REPORTS / name, 'w', newline='', encoding='utf-8') as fh:
        w = csv.DictWriter(fh, fieldnames=fields); w.writeheader(); w.writerows(rows)
write_csv('evaluation_summary.csv', [dict(model_version=MODEL_VERSION, split=s, **v) for s, v in EVAL.items()])
write_csv('per_class_metrics.csv', [dict(split=s, class_id=0, class_name='litter', precision=v['precision'],
                                         recall=v['recall'], f1=v['f1'], mAP50=v['mAP50'], mAP50_95=v['mAP50_95'],
                                         gt_boxes=v['boxes']) for s, v in EVAL.items()])
import pandas as pd
res = pd.read_csv(RUN_DIR / 'results.csv'); res.columns = [c.strip() for c in res.columns]
fit = 0.1 * res['metrics/mAP50(B)'] + 0.9 * res['metrics/mAP50-95(B)']
bi = int(fit.idxmax())
gpu_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu'
TRAIN_SUMMARY = dict(model_version=MODEL_VERSION, run_name=RUN_NAME, started_from=START_PATH.name, start_sha256=got,
                     epochs_requested=EPOCHS, epochs_run=len(res), best_epoch=int(res['epoch'][bi]), imgsz=IMG_SIZE,
                     batch=BATCH, optimizer=OPTIMIZER, lr0=LR0, patience=PATIENCE, seed=SEED, device=gpu_name,
                     train_minutes=TRAIN_MIN, ultralytics=ultralytics.__version__, torch=torch.__version__,
                     val_mAP50_best_epoch=round(float(res['metrics/mAP50(B)'][bi]), 4),
                     val_mAP50_95_best_epoch=round(float(res['metrics/mAP50-95(B)'][bi]), 4),
                     smoke_test=SMOKE, trained_at=dt.datetime.now(dt.timezone.utc).isoformat(timespec='seconds'))
write_csv('training_summary.csv', [TRAIN_SUMMARY])
print(TRAIN_SUMMARY)

In [ ]:
# ===================================================== 11. Per-detection test report: prediction_results, false_positives, false_negatives
import numpy as np
def iou(a, b):
    ix = max(0, min(a[2], b[2]) - max(a[0], b[0])); iy = max(0, min(a[3], b[3]) - max(a[1], b[1]))
    inter = ix * iy
    u = (a[2] - a[0]) * (a[3] - a[1]) + (b[2] - b[0]) * (b[3] - b[1]) - inter
    return inter / u if u > 0 else 0.0
def in_ignore(box, regions):
    cx, cy = (box[0] + box[2]) / 2, (box[1] + box[3]) / 2
    return any(r[0] <= cx <= r[2] and r[1] <= cy <= r[3] for r in regions)
pred_rows, fp_rows, fn_rows = [], [], []
for p in split_files('test'):
    r = best.predict(str(p), imgsz=IMG_SIZE, conf=EVAL_CONF, device=DEVICE, verbose=False)[0]
    h, w = r.orig_shape
    gts = gt_boxes(p, w, h)
    used = set()
    preds = sorted(zip(r.boxes.xyxy.cpu().numpy().tolist(), r.boxes.conf.cpu().numpy().tolist()), key=lambda t: -t[1])
    for k, (b, c) in enumerate(preds):
        best_j, best_iou = None, 0.0
        for j, g in enumerate(gts):
            if j not in used and (v := iou(b, g)) > best_iou:
                best_j, best_iou = j, v
        if best_j is not None and best_iou >= MATCH_IOU:
            used.add(best_j); outcome = 'TP'
        elif in_ignore(b, IGNORE.get(p.name, [])):
            outcome = 'IGNORED_REGION'
        else:
            outcome = 'FP'
        row = dict(image=p.name, pred_index=k, class_name='litter', confidence=round(c, 4),
                   x1=round(b[0], 1), y1=round(b[1], 1), x2=round(b[2], 1), y2=round(b[3], 1),
                   matched_gt_index=best_j if outcome == 'TP' else '', iou=round(best_iou, 3), outcome=outcome)
        pred_rows.append(row)
        if outcome == 'FP':
            fp_rows.append(row)
    for j, g in enumerate(gts):
        if j not in used:
            fn_rows.append(dict(image=p.name, gt_index=j, class_name='litter', x1=round(g[0], 1), y1=round(g[1], 1),
                                x2=round(g[2], 1), y2=round(g[3], 1),
                                min_side_px=round(min(g[2] - g[0], g[3] - g[1]), 1)))
PRED_FIELDS = ['image', 'pred_index', 'class_name', 'confidence', 'x1', 'y1', 'x2', 'y2', 'matched_gt_index', 'iou', 'outcome']
write_csv('prediction_results.csv', pred_rows, PRED_FIELDS)
write_csv('false_positives.csv', fp_rows, PRED_FIELDS)
write_csv('false_negatives.csv', fn_rows, ['image', 'gt_index', 'class_name', 'x1', 'y1', 'x2', 'y2', 'min_side_px'])
tp = sum(r['outcome'] == 'TP' for r in pred_rows)
print(f'test @conf {EVAL_CONF}: TP {tp}  FP {len(fp_rows)}  FN {len(fn_rows)}  ignored {sum(r["outcome"] == "IGNORED_REGION" for r in pred_rows)}')

In [ ]:
# ===================================================== 12. model_comparison.csv (test scene, class-agnostic)
rows = [dict(model='%s (new, 1 class)' % MODEL_VERSION, split='test', **{k: EVAL['test'][k] for k in
             ('precision', 'recall', 'f1', 'mAP50', 'mAP50_95')}, note='trained in this notebook')]
v3 = pathlib.Path(os.environ.get('LDM_V3_WEIGHTS') or DRIVE / V3_WEIGHTS)
if v3.is_file():
    m = YOLO(str(v3)).val(data=str(DATA_YAML), split='test', imgsz=IMG_SIZE, batch=BATCH, device=DEVICE,
                          single_cls=True, plots=False, project=str(RUN_DIR), name='eval_v3_test', exist_ok=True,
                          conf=0.001, iou=0.6)
    p, r = float(m.box.mp), float(m.box.mr)
    rows.append(dict(model='v3_store_cctv (production, 10 classes merged)', split='test', precision=round(p, 4),
                     recall=round(r, 4), f1=round(2 * p * r / (p + r), 4) if p + r else 0.0,
                     mAP50=round(float(m.box.map50), 4), mAP50_95=round(float(m.box.map), 4),
                     note='class-agnostic: every v3 class counted as litter'))
else:
    print('v3 weights not on Drive: comparison has the new model only')
write_csv('model_comparison.csv', rows)
pd.DataFrame(rows)

In [ ]:
# ===================================================== 13. Package for LDM (separate version folder; never activated)
PKG = RUN_DIR / 'package' / MODEL_VERSION
if PKG.exists():
    shutil.rmtree(PKG.parent)
PKG.mkdir(parents=True)
shutil.copy2(BEST, PKG / 'best.pt'); shutil.copy2(LAST, PKG / 'last.pt')
for f in ('results.csv', 'args.yaml', 'results.png', 'confusion_matrix.png', 'confusion_matrix_normalized.png',
          'BoxPR_curve.png', 'BoxF1_curve.png'):
    if (RUN_DIR / f).is_file(): shutil.copy2(RUN_DIR / f, PKG / f)
for s in ('val', 'test'):
    for f in ('confusion_matrix.png', 'BoxPR_curve.png'):
        if (RUN_DIR / f'eval_{s}' / f).is_file(): shutil.copy2(RUN_DIR / f'eval_{s}' / f, PKG / f'{s}_{f}')
shutil.copytree(REPORTS, PKG / 'reports')
for f in ('dataset_manifest.csv', 'annotation_audit.csv', 'dataset_summary.csv', 'class_distribution.csv', 'split_summary.csv'):
    if (DS / f).is_file(): shutil.copy2(DS / f, PKG / 'reports' / f)
(PKG / 'data.yaml').write_text((DS / 'data.yaml').read_text())
(PKG / 'class_names.json').write_text(json.dumps(['litter'], indent=2))
meta = dict(version=MODEL_VERSION, name=MODEL_VERSION, trained_at=TRAIN_SUMMARY['trained_at'], base_model=START_PATH.name,
            imgsz=IMG_SIZE, classes=['litter'],
            metrics=dict(epoch=TRAIN_SUMMARY['best_epoch'], epochs_run=TRAIN_SUMMARY['epochs_run'], **EVAL['val'],
                         source='ultralytics val split (same scene as train: optimistic)'),
            evaluation=dict(test=EVAL['test'], test_note='independent synthetic scene (B_roadside_bin)'),
            notes=('One-class litter model trained on LDM_TRAINING_READY (AUG100, synthetic, 2 scenes). NOT a '
                   'replacement for the 10-class v3_store_cctv. Candidate only: activate manually after review.'
                   + (' SMOKE TEST - NOT A REAL MODEL.' if SMOKE else '')),
            status='candidate_not_active', smoke_test=SMOKE, training=TRAIN_SUMMARY)
(PKG / 'model_meta.json').write_text(json.dumps(meta, indent=2))
sums = [f'{sha256(p)}  {p.relative_to(PKG).as_posix()}' for p in sorted(PKG.rglob('*')) if p.is_file()]
(PKG / 'SHA256SUMS').write_text('\n'.join(sums) + '\n')
ZIP = shutil.make_archive(str(RUN_DIR / f'ldm_model_{MODEL_VERSION}'), 'zip', root_dir=PKG.parent, base_dir=MODEL_VERSION)
print('package:', ZIP)
if IN_COLAB:
    from google.colab import files
    files.download(ZIP)

## Install in LDM (on the PC) — the production model is not replaced
```
python -m training.install_trained_model --package "C:\path\to\ldm_model_v4_aug100_litter1c.zip"
```
* Copies into `models/litter_yolo/<MODEL_VERSION>/` (refuses an existing folder and the name `v3_store_cctv`), checks SHA256SUMS and the class list.
* After the next backend restart it appears in **Image Analysis → Model version** as a non-active option; analyses run with it say so in their CSV metadata.
* **Activation** is a separate, manual admin action (Models page). **Rollback:** re-activate `v3_store_cctv` on the Models page (its weights are never changed), or delete the new folder while it is not active.